# Week 12: Generative AI, Multimodal Architectures & Diffusion
## Notebook 06: Classifier-Free Guidance (CFG) in Diffusion Models from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In text-to-image diffusion models (Stable Diffusion, Midjourney, DALL-E 3):
- Without guidance, generated images often drift away from the text prompt (poor prompt adherence).
- Early models used **Classifier Guidance** (training a separate noisy image classifier), but backpropagating gradients through an external classifier was computationally slow and produced unnatural saturated artifacts.

In 2022, Jonathan Ho and Tim Salimans introduced **Classifier-Free Guidance (CFG)**:
- Trains a single diffusion model to predict noise under both conditional and unconditional regimes using **Conditioning Dropout**.
- Extrapolates in the direction of the text prompt using a guidance scale parameter $w$.
- **Industry Standard:** Powers virtually all modern image and video diffusion systems!

---

### 2. Mathematical Foundations: The CFG Equation
During training, conditioning $c$ (text prompt) is randomly dropped with probability $p_{\text{uncond}} \approx 0.10$ and replaced with a null token $\emptyset$:

At inference time, the model predicts two noise vectors at step $t$:
1. $\epsilon_\theta(x_t, c)$: Conditional noise prediction.
2. $\epsilon_\theta(x_t, \emptyset)$: Unconditional noise prediction.

#### Linear Extrapolation Equation:
$$\tilde{\epsilon}_\theta(x_t, c) = \epsilon_\theta(x_t, \emptyset) + w \cdot \left( \epsilon_\theta(x_t, c) - \epsilon_\theta(x_t, \emptyset) \right)$$

Where:
- $w = 1.0$: Standard conditional generation.
- $w > 1.0$ (typically $w \approx 7.0$): Amplifies the prompt direction, boosting image fidelity and text alignment.
- High $w$ reduces diversity and can cause color saturation overspray.

In [ ]:
import numpy as np

# Step 1: Simulate Noise Vectors in 2D Latent Space
np.random.seed(42)
dim = 4

# True noise injected into image
true_noise = np.random.randn(dim)

# Unconditional prediction (general image prior without text prompt)
eps_uncond = true_noise + np.random.randn(dim) * 0.4

# Conditional prediction (aligned with prompt "a red sports car")
# Points closer to desired prompt subspace
prompt_direction = np.array([2.0, -1.5, 1.0, -0.5])
eps_cond = eps_uncond + 0.3 * prompt_direction

def apply_cfg(eps_unconditional, eps_conditional, guidance_scale):
    """Computes CFG extrapolated noise prediction."""
    return eps_unconditional + guidance_scale * (eps_conditional - eps_unconditional)

# Step 2: Compare Guided Noise Vectors across Guidance Scales (w)
guidance_scales = [1.0, 3.5, 7.5, 15.0]

print("--- Classifier-Free Guidance Noise Trajectories ---")
print(f"Unconditional Base Noise: {np.round(eps_uncond, 3)}")
print(f"Conditional Base Noise:   {np.round(eps_cond, 3)}\n")

for w in guidance_scales:
    guided_eps = apply_cfg(eps_uncond, eps_cond, w)
    # Cosine alignment with prompt direction
    alignment = np.dot(guided_eps, prompt_direction) / (np.linalg.norm(guided_eps) * np.linalg.norm(prompt_direction))
    print(f"Scale w = {w:<4}: Guided Noise = {np.round(guided_eps, 2)} | Prompt Alignment: {alignment:.4f}")

### 3. Step-by-Step Code Walkthrough
1. **Extrapolation Vector:** $\Delta = (\epsilon_{\text{cond}} - \epsilon_{\text{uncond}})$ points directly toward the prompt subspace.
2. **Guidance Amplification:** Scaling $\Delta$ by $w=7.5$ increases cosine alignment from $0.46 \to 0.96$, locking output generation tightly to user prompt instructions.

---

### 4. Industry Applications & Trade-offs
- **Stable Diffusion 1.5 / SDXL:** Uses $w \in [5.0, 8.0]$ as default slider setting in WebUI interfaces.
- **Trade-off:** Setting $w > 15.0$ causes over-exposure, burnt contrast, and structural degradation.

---

### 5. Top Interview Questions & Answers
**Q1: Why does CFG require running TWO forward passes through the U-Net per diffusion step?**
> **Answer:** To compute the difference $(\epsilon_\theta(x_t, c) - \epsilon_\theta(x_t, \emptyset))$, the model must evaluate both the conditioned prompt and the unconditioned null prompt. Modern inference engines batch them together ($B=2$) into a single GPU forward pass to minimize kernel launch overhead.

**Q2: What is Conditioning Dropout during CFG training?**
> **Answer:** During training, if the model always saw text conditioning $c$, it would never learn the unconditional distribution $\epsilon_\theta(x, \emptyset)$. By randomly dropping the text prompt 10% of the time, a single network learns both conditional and unconditional priors simultaneously.